# งาน RAG Chatbot — PC-Fix: ผู้ช่วยแก้ปัญหา Hardware คอมพิวเตอร์

**หัวข้อที่เลือก:** ถาม-ตอบปัญหา Hardware คอมพิวเตอร์ (ไทย/อังกฤษ) จากคลังเอกสาร 12 ไฟล์

**URL หน้าเว็บ Streamlit:** `<ใส่ลิงก์หลัง Deploy>`

**GitHub Repository URL:** `<ใส่ลิงก์ Repo>`

โครงสร้าง Repo: `app.py`, `requirements.txt`, `README.md`, `data/` (12 ไฟล์), `test_questions.csv`

ส่วนด้านล่างคือการทดลอง pipeline RAG แบบย่อใน Colab (ตรงกับตรรกะใน app.py)

In [ ]:
!pip -q install sentence-transformers faiss-cpu groq pythainlp

In [ ]:
# 1) Document Loading & Chunking (ตัวอย่างเอกสารย่อ — ใน Repo จริงใช้ไฟล์ใน data/)
import re
docs = {
 "01_no_power.txt": "คอมพิวเตอร์เปิดไม่ติด: ตรวจสายไฟ AC และสวิตช์หลัง PSU ให้อยู่ตำแหน่ง I ถอดปลั๊กแล้วกดปุ่ม Power ค้าง 10-15 วินาที",
 "03_overheat.txt": "CPU ขณะโหลดหนักไม่ควรเกิน 80-85 องศาเซลเซียส หากสูงกว่า 90 องศาถือว่าอันตราย เครื่องจะลดความเร็วอัตโนมัติ (Thermal Throttling)",
 "10_bsod_en.txt": "WHEA_UNCORRECTABLE_ERROR is a hardware error often caused by overheating, unstable overclocking, a failing PSU or faulty RAM.",
}
def chunk(text, size=500, overlap=80):
    text = re.sub(r"\s+", " ", text).strip()
    return [text[i:i+size] for i in range(0, len(text), size-overlap)]
chunks = [{"source": k, "text": c} for k, v in docs.items() for c in chunk(v)]
len(chunks)

In [ ]:
# 2) Embedding & FAISS
import faiss, numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb = model.encode([c["text"] for c in chunks], normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
def retrieve(q, k=2):
    s, i = index.search(model.encode([q], normalize_embeddings=True).astype("float32"), k)
    return [{**chunks[j], "score": float(x)} for x, j in zip(s[0], i[0])]
retrieve("CPU ร้อนแค่ไหนอันตราย")

In [ ]:
# 3-4) Prompt + Groq LLM (ใส่คีย์ผ่าน Colab Secrets ชื่อ GROQ_API_KEY — ห้ามเขียนคีย์ลงในโค้ด)
from google.colab import userdata
from groq import Groq
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
SYSTEM = "ตอบจาก [CONTEXT] เท่านั้น อ้างอิง [ชื่อไฟล์] ถ้าไม่มีข้อมูลให้ตอบว่า 'ไม่พบข้อมูลในเอกสาร'"
def ask(q):
    hits = retrieve(q)
    ctx = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    r = client.chat.completions.create(model="llama-3.3-70b-versatile", temperature=0.1,
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":f"[CONTEXT]\n{ctx}\n\n[QUESTION]\n{q}"}])
    return r.choices[0].message.content, hits
print(ask("CPU ร้อนแค่ไหนอันตราย")[0]); print(ask("ราคา RTX 5090 เท่าไร")[0])